# importing libraries

In [62]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# importing dataset

In [63]:
iris = load_iris()
X = iris.data
y = iris.target

# pre-processing

## encoding categorical target

In [64]:
encoder = OneHotEncoder(sparse_output=False)
y = y.reshape(-1, 1)
y = encoder.fit_transform(y)

## splitting

In [65]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## scaling

In [66]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# vectorizing data

In [67]:
X_train = X_train.T
y_train = y_train.T
X_test = X_test.T
y_test = y_test.T

# activation function

## for hidden layers

In [68]:
def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))

def sigmoidDerivative(Z):
    s = sigmoid(Z)
    return s * (1 - s)

def relu(Z):
    return np.maximum(0, Z)

def reluDerivative(Z):
    return np.where(Z > 0, 1, 0)

def activation(Z, activationType):
    if activationType == "sigmoid":
        return sigmoid(Z)
    elif activationType == "relu":
        return relu(Z)

def activationDerivative(Z, activationType):
    if activationType == "sigmoid":
        return sigmoidDerivative(Z)
    elif activationType == "relu":
        return reluDerivative(Z)

## for output layer

In [69]:
def softmax(Z):
    expZ = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    return expZ / np.sum(expZ, axis=0, keepdims=True)

# Neural Network

In [70]:
class NeuralNetwork:
    def __init__(self, inputLayerSize, hiddenLayerSize, outputLayerSize, activationType):
        # initializing random weights and zero biases
        self.W1 = np.random.randn(hiddenLayerSize, inputLayerSize) * 0.01
        self.b1 = np.zeros((hiddenLayerSize, 1))
        self.W2 = np.random.randn(outputLayerSize, hiddenLayerSize) * 0.01
        self.b2 = np.zeros((outputLayerSize, 1))
        self.activationType = activationType

    def forwardPass(self, X):
        # Hidden layer
        Z1 = np.dot(self.W1, X) + self.b1
        A1 = activation(Z1, self.activationType)

        # Output layer
        Z2 = np.dot(self.W2, A1) + self.b2
        A2 = softmax(Z2)

        # Store intermediate values in a cache for backpropagation.
        cache = (Z1, A1, Z2, A2)
        return A2, cache

    def backwardProp(self, X, y, cache):
        m = X.shape[1] # Number of samples
        Z1, A1, Z2, A2 = cache

        # Output layer gradients
        dZ2 = A2 - y # The derivative of Cross-Entropy loss
        dW2 = (1/m) * np.dot(dZ2, A1.T)
        db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)

        # Hidden layer gradients
        dZ1 = np.dot(self.W2.T, dZ2) * activationDerivative(Z1, self.activationType)
        dW1 = (1/m) * np.dot(dZ1, X.T)
        db1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)

        gradients = (dW1, db1, dW2, db2)
        return gradients

    # update weights and biases
    def updateParams(self, gradients, lr):
        dW1, db1, dW2, db2 = gradients
        self.W1 -= lr * dW1
        self.b1 -= lr * db1
        self.W2 -= lr * dW2
        self.b2 -= lr * db2

    # training
    def train(self, X, y, epochs, lr):
        m = X.shape[1]
        for i in range(epochs):
            A2, cache = self.forwardPass(X)
            gradients = self.backwardProp(X, y, cache)
            self.updateParams(gradients, lr)

    # for evaluating
    def predict(self, X):
        A2, _ = self.forwardPass(X)
        predictions = np.argmax(A2, axis=0)
        return predictions


In [71]:
if __name__ == "__main__":
    # model parameters
    inputLayerSize = X_train.shape[0]
    hiddenLayerSize = 10 # Number of neurons in the hidden layer
    outputLayerSize = y_train.shape[0]

    # Hyperparameters
    epochs = 1000
    lr = 0.05

    activationType = "sigmoid"
    activationType2 = "relu"

    # model = NeuralNetwork(inputLayerSize, hiddenLayerSize, outputLayerSize, activationType)
    model = NeuralNetwork(inputLayerSize, hiddenLayerSize, outputLayerSize, activationType2)

    # training
    model.train(X_train, y_train, epochs, lr)

    # predictions on test data
    testPredictions = model.predict(X_test)
    testLabels = np.argmax(y_test, axis=0)

    # accuracy
    accuracy = np.mean(testPredictions == testLabels) * 100
    print(f"\nModel Accuracy on the test set: {accuracy}%")


Model Accuracy on the test set: 100.0%
